# SOMocluSummarizer + Quality Control demo -- KiDS-Legacy gold-weight version

**Original author:** Ziang Yan

**References:**
- Wright et al. 2020, quality-control criteria QC1/QC2 ([arXiv:2007.15635](https://arxiv.org/pdf/2007.15635))
- Stölzner et al. 2025 (KiDS-Legacy redshift calibration, gold weight), [arXiv:2503.19440](https://arxiv.org/abs/2503.19440)
- Wright et al. 2025 (KiDS-Legacy cosmic shear), [arXiv:2503.19441](https://arxiv.org/abs/2503.19441)

This notebook creates an end-to-end example for the SOM summarizer plus quality control defined in [arXiv:2007.15635](https://arxiv.org/pdf/2007.15635):

1) create photometric realizations for a training and spectroscopic sample;
2) measure BPZ for the training and spectroscopic samples;
3) make the same tomographic cut on the training and spec samples;
4) inform a `rail_som` model with the training sample and summarize it with the spec sample;
5) perform quality control ([arXiv:2007.15635](https://arxiv.org/pdf/2007.15635));
6) summarize the goodness of redshift calibration and compare between QCs.

**KiDS-Legacy gold-weight quality control:** the traditional (KiDS-1000-style) approach trains a single SOM and applies each quality-control criterion as a hard cut on the SOM clusters (a cluster either passes QC1/QC2 or it doesn't, under that one SOM training). Here, following Stölzner et al. 2025 ([arXiv:2503.19440](https://arxiv.org/abs/2503.19440)), we instead train `N_REALIZATIONS = 10` independent SOM realizations (all on the **same** training catalog; only the SOM's random initialization differs between realizations). For each realization we evaluate the **same QC1/QC2 criteria** on that realization's own clustering, and record, for every target galaxy, whether its cluster passed QC1, QC2, and QC1+QC2 in that realization. The **gold weight** for each QC scheme is then the fraction of the 10 realizations in which a galaxy's cluster passed that criterion -- a continuous 0-1 weight that replaces the hard `useful_clusters` cut, exactly analogous to how `11.5_SomocluSOM_KL.ipynb` replaces the plain gold-class cut from `11_SomocluSOM.ipynb`. All of the quantitative QC comparisons (fiducial vs. QC1 vs. QC2 vs. QC1+QC2, N(z) plots, bias box plot) are built from this gold-weighted calibration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import pickle
import rail
import os
import qp
from rail.core.utils import RAILDIR

import tables_io
from rail.core.data import Hdf5Handle, TableHandle, ModelHandle
from rail.core.stage import RailStage
from rail.estimation.algos.somoclu_som import SOMocluInformer, SOMocluSummarizer
from rail.estimation.algos.somoclu_som import get_bmus, plot_som, _computemagcolordata
import sklearn.cluster as sc
from scipy.stats import median_abs_deviation

First, let's grab some data files.  For the SOM, we will want to train on a fairly large, representative set that encompasses all of our expected data.  We'll grab a larger data file than we typically use in our demos to ensure that we construct a meaningful SOM.

## Run this command on the command line to get the larger data file to train the SOM:
`curl -O https://portal.nersc.gov/cfs/lsst/schmidt9/healpix_10326_bright_data.hdf5`

and then move the resulting file to this directory, i.e. RAIL/examples/estimation.  This data consists of ~150,000 galaxies from a single healpix pixel of the comsoDC2 truth catalog with mock 10-year magnitude errors added.  It is cut at a relatively bright i<23.5 magnitudes in order to concentrate on galaxies with particularly high S/N rates.

# First read the target and spec catalogue from a pre-trained pzflow stage.

In [ ]:
training_file = "./healpix_10326_bright_data.hdf5"

if not os.path.exists(training_file):
  os.system('curl -O https://portal.nersc.gov/cfs/lsst/PZ/healpix_10326_bright_data.hdf5')


In [ ]:
training_data = tables_io.read(training_file)

In [ ]:
pmask = (training_data['photometry']['mag_i_lsst'] <23.5)
trim_test = {}
for key in training_data['photometry'].keys():
    trim_test[key] = training_data['photometry'][key][pmask]
trim_dict = dict(photometry=trim_test)
target_data_all = Hdf5Handle("target_data_raw",data=trim_dict)

In [ ]:
from rail.utils.path_utils import find_rail_file

specfile = find_rail_file("examples_data/testdata/test_dc2_validation_9816.hdf5")
ref_data_raw = tables_io.read(specfile)['photometry']
smask = (ref_data_raw['mag_i_lsst'] <23.5)
trim_spec = {}
for key in ref_data_raw.keys():
    trim_spec[key] = ref_data_raw[key][smask]
trim_dict = dict(photometry=trim_spec)
ref_data_all = Hdf5Handle("ref_data_raw", data=trim_dict)

# Now measure the photometric redshifts using the `bpz_lite`

In [ ]:
bands = ["u", "g", "r", "i", "z", "y"]
lsst_bands = []
lsst_errs = []
lsst_filts = []
for band in bands:
    lsst_bands.append(f"mag_{band}_lsst")
    lsst_errs.append(f"mag_err_{band}_lsst")
    lsst_filts.append(f"DC2LSST_{band}")
print(lsst_bands)
print(lsst_filts)

In [ ]:
from rail.core.utils import RAILDIR
import os
from rail.core.utils import RAILDIR
from rail.estimation.algos.bpz_lite import BPZliteInformer, BPZliteEstimator
from rail.core.data import ModelHandle
custom_data_path = RAILDIR + '/rail/examples_data/estimation_data/data'

hdfnfile = os.path.join(RAILDIR, "rail/examples_data/estimation_data/data/CWW_HDFN_prior.pkl")
sedfile = os.path.join(RAILDIR, "rail/examples_data/estimation_data/data/SED/COSMOS_seds.list")

with open(hdfnfile, "rb") as f:
    hdfnmodel = pickle.load(f)

custom_dict_phot = dict(hdf5_groupname="photometry",
                   output="bpz_results_phot_qc_KL.hdf5",
                   bands=lsst_bands,
                   err_bands=lsst_errs,
                   filter_list=lsst_filts,
                   prior_band='mag_i_lsst',spectra_file=sedfile,
                   data_path=custom_data_path,
                   no_prior=False)

custom_dict_spec = dict(hdf5_groupname="photometry",
                   output="bpz_results_spec_qc_KL.hdf5",
                    bands=lsst_bands,
                   err_bands=lsst_errs,
                   filter_list=lsst_filts,
                   prior_band='mag_i_lsst',spectra_file=sedfile,
                   data_path=custom_data_path,
                   no_prior=False)

cosmospriorfile = os.path.join(RAILDIR, "rail/examples_data/estimation_data/data/COSMOS31_HDFN_prior.pkl")
cosmosprior = ModelHandle("cosmos_prior",path=cosmospriorfile)

phot_run = BPZliteEstimator.make_stage(name="rerun_bpz_phot_kl", model=cosmosprior, **custom_dict_phot)
spec_run = BPZliteEstimator.make_stage(name="rerun_bpz_spec_kl", model=cosmosprior, **custom_dict_spec)

In [ ]:
from collections import OrderedDict
phot_run.estimate(target_data_all)

In [ ]:
spec_run.estimate(ref_data_all)

In [ ]:
phot_bpz_file = 'bpz_results_phot_qc_KL.hdf5'
bpz_phot_all = tables_io.read(phot_bpz_file)['ancil']['zmode']

spec_bpz_file = 'bpz_results_spec_qc_KL.hdf5'
bpz_spec_all = tables_io.read(spec_bpz_file)['ancil']['zmode']

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(target_data_all.data['photometry']['redshift'], bpz_phot_all, s=0.3)
plt.plot(target_data_all.data['photometry']['redshift'],target_data_all.data['photometry']['redshift'], color='C1')
plt.title('Test data')
plt.xlabel(r'$Z_{\mathrm{spec}}$', fontsize=15)
plt.ylabel(r'$Z_{\mathrm{phot}}$', fontsize=15)

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(ref_data_all.data['photometry']['redshift'], bpz_spec_all, s=0.3)
plt.plot(ref_data_all.data['photometry']['redshift'],ref_data_all.data['photometry']['redshift'], color='C1')
plt.title('Spec data')
plt.xlabel(r'$Z_{\mathrm{spec}}$', fontsize=15)
plt.ylabel(r'$Z_{\mathrm{phot}}$', fontsize=15)

## cut the data to make a tomographic bin

In [ ]:
bin_low = 0.2
bin_high = 0.5

In [ ]:
trim_data_test = {}

mask_phot = ((bpz_phot_all > bin_low) & (bpz_phot_all < bin_high))
mask_phot &= (target_data_all.data['photometry']['redshift'] > 0)

bpz_phot = bpz_phot_all[mask_phot]

for key in target_data_all.data['photometry'].keys():
    trim_data_test[key] = target_data_all.data['photometry'][key][mask_phot]
trimdict_test = dict(photometry=trim_data_test)
target_data = Hdf5Handle("testing_data",data=trimdict_test)

In [ ]:
trim_data_spec = {}

mask_spec = ((bpz_spec_all > bin_low) & (bpz_spec_all<bin_high))
mask_spec &= (ref_data_all.data['photometry']['redshift'] > 0)

bpz_spec = bpz_spec_all[mask_spec]

for key in target_data_all.data['photometry'].keys():
    trim_data_spec[key] = ref_data_all.data['photometry'][key][mask_spec]
trimdict_spec = dict(photometry=trim_data_spec)
ref_data = Hdf5Handle("ref_data", data=trimdict_spec)


In [ ]:
plt.title('Redshift distributions')
plt.xlabel(r'$Z_{\mathrm{spec}}$')
plt.ylabel('dN/dz')

plt.hist(target_data.data['photometry']['redshift'], bins=50, density=True, histtype='step', label='Target')
plt.hist(ref_data.data['photometry']['redshift'], bins=50, density=True, histtype='step', label='Reference')
plt.axvline(bin_low, color='k', linestyle='--')
plt.axvline(bin_high, color='k', linestyle='--')
plt.legend()

# Now let's train the SOM with the color from the target set

We need to define all of our necessary initialization params, which includes the following:
- `name` (str): the name of our estimator, as utilized by ceci
- `model` (str): the name for the model file containing the SOM and associated parameters that will be written by this stage
- `hdf5_groupname` (str): name of the hdf5 group (if any) where the photometric data resides in the training file
- `n_rows` (int): the number of dimensions in the y-direction for our 2D SOM
- `n_columns` (int): the number of dimensions in the x-direction for our 2D SOM
- `grid_type` (str): the parameter that specifies the grid form of the nodes. Options: `rectangular`(default) and `hexagonal`.
- `initialization` (str): the parameter specifying the method of initializing the SOM. Options: `pca`: principal componant analysis (default); `random`: randomly initialize the SOM.
- `maptype` (str): the parameter specifying the map topology. Options: `planar`(default) and `toroid`.
- `n_epochs` (int): the number of iteration steps during SOM training.
- `std_coeff` (float): the "radius" of how far to spread changes in the SOM
- `som_learning_rate` (float): a number between 0 and 1 that controls how quickly the weighting function decreases.
- `column_usage` (str): determines what values are used to construct the SOM; we use `colors`.

**KiDS-Legacy addition:** instead of informing a single SOM, we train `N_REALIZATIONS = 10` SOMs, all on the exact same `target_data`. The only thing that differs between realizations is the random codebook initialization (`initialization='random'`): unlike the `rail_som` version used by `11_SomocluSOM.ipynb`/`11.5_SomocluSOM_KL.ipynb` (which silently hardcodes PCA initialization), the `rail_som` version used by this notebook's kernel properly passes `initialization` through to `Somoclu(...)`, so `initialization='random'` genuinely gives each realization an independent random starting codebook, exactly as intended. We additionally seed numpy's global RNG differently per realization as a safeguard, though this `rail_som` version's `SOMocluInformer.run()` does not itself consume the `seed` config parameter.

In [ ]:
data_size = training_data['photometry']['mag_i_lsst'].shape[0]
dim = np.sqrt(5 * np.sqrt(data_size))  # sqrt(5) * sqrt(N) = sqrt(5N)
dim = int(dim)  # convert to integer for SOM dimensions
if dim % 2 == 0:
    dim += 1  # ensure odd dimensions for hexagonal grid
grid_type = 'hexagonal'
N_REALIZATIONS = 10  # N_repl in Stolzner et al. 2025 (arXiv:2503.19440)

# base configuration shared by every SOM realization; `model` and `seed`
# are set individually for each realization below
base_inform_dict = dict(hdf5_groupname='photometry',
                   n_rows=dim, n_columns=dim,
                   grid_type=grid_type,
                   maptype='toroid',
                   initialization='random',
                   n_epochs=30,
                   std_coeff=12.0, som_learning_rate=0.75,
                   column_usage='colors')

In [ ]:
%%time
som_models = []
rng_master = np.random.default_rng(42)

for j in range(N_REALIZATIONS):
    seed_j = int(rng_master.integers(0, 1_000_000))
    np.random.seed(seed_j)

    inform_dict_j = dict(base_inform_dict)
    inform_dict_j['model'] = f'output_SOMoclu_model_KL_real{j}.pkl'
    inform_dict_j['seed'] = seed_j

    inform_som_j = SOMocluInformer.make_stage(name=f'inform_som_real{j}', **inform_dict_j)
    inform_som_j.inform(target_data)
    som_models.append(inform_som_j.model)
    print(f"Trained SOM realization {j + 1}/{N_REALIZATIONS} (same training set, seed={seed_j})")

We designate the **first** realization as our fiducial SOM: it is used below for the illustrative mean-redshift and QC maps, and as the SOM passed to `SOMocluSummarizer` to build the final N(z). The gold weight for each QC scheme, however, is computed by combining information from *all* `N_REALIZATIONS` SOMs.

In [ ]:
model = som_models[0]
SOM = model['som']
usecols = model['usecols']
ref_column_name = model['ref_column']
column_usage = model['column_usage']
n_rows = model['n_rows']
n_columns = model['n_columns']

We can calculate the best SOM cell using the `get_bmus()` function defined in `somoclu_som.py`, which will return the 2D SOM coordinates for each galaxy. We then group the SOM cells into hierarchical clusters and calculate the occupation and mean redshift in each cluster -- except we do this once per SOM realization, since the color features themselves don't depend on which realization we're looking at (they only depend on the reference/target photometry), only the best-matching cell (and hence cluster) assignment does.

In [ ]:
bands = ['u','g','r','i','z','y']
bandnames = [f"mag_{band}_lsst" for band in bands]

ngal_ref = len(ref_data.data['photometry']['mag_i_lsst'])
ngal_target = len(target_data.data['photometry']['mag_i_lsst'])

# colors only depend on the photometry, not on which SOM realization we use,
# so we can compute them once and reuse them for every realization below
ref_colors = _computemagcolordata(ref_data.data['photometry'], ref_column_name, usecols, column_usage)
target_colors = _computemagcolordata(target_data.data['photometry'], ref_column_name, usecols, column_usage)

The next cells define some functions to plot cluster boundaries on a SOM grid.

In [ ]:
def plot_cluster_boundaries(ax, SOM, n_clusters, cluster_inds=None, topology='hexagonal'):
    dim = SOM.codebook.shape[0]
    som_cluster_ind = SOM.clusters.reshape(-1)
    som_centers = find_cell_centers(dim, topology=topology)
    som_centers_l = np.array([som_centers.T[0]-dim*1, som_centers.T[1]]).T
    som_centers_r = np.array([som_centers.T[0]+dim*1, som_centers.T[1]]).T
    som_centers_u = np.array([som_centers.T[0], som_centers.T[1]+dim*np.sqrt(3)/2]).T
    som_centers_b = np.array([som_centers.T[0], som_centers.T[1]-dim*np.sqrt(3)/2]).T
    if cluster_inds is None:
        cluster_inds = np.arange(n_clusters)
    for i in (cluster_inds):
        centers = (np.vstack([som_centers[np.where(som_cluster_ind==i)[0]],
                         som_centers_l[np.where(som_cluster_ind==i)[0]], som_centers_u[np.where(som_cluster_ind==i)[0]],
                         som_centers_r[np.where(som_cluster_ind==i)[0]], som_centers_b[np.where(som_cluster_ind==i)[0]]]))
        linep = get_manycells_boundary(centers, topology='hexagonal')
        for points in linep:
            if points.T[0].min() < som_centers.T[0].min()-1 or points.T[0].max() > som_centers.T[0].max()+1 or points.T[1].min() < som_centers.T[1].min()-np.sqrt(3)/2 or points.T[1].max() > som_centers.T[1].max()+np.sqrt(3)/2:
            #plt.plot(points.T[0], points.T[1], color='blue')
                continue
            ax.plot(points.T[0], points.T[1], color='k', lw=0.2)
    return

def find_cell_centers(dim, topology='rectangular'):
    if topology == 'rectangular':
        x = np.arange(dim) + 0.5
        y = np.arange(dim) + 0.5
        xx, yy = np.meshgrid(x, y)
        xx = xx.reshape(-1)
        yy = yy.reshape(-1)
        centers = np.array([xx, yy]).T
    if topology == 'hexagonal':
        yy, xx= np.meshgrid(np.arange(dim), np.arange(dim))
        shift = np.zeros(dim)
        shift[::2]=-0.5
        xx = xx + shift
        yy = yy * (np.sqrt(3) / 2)
        centers = np.array([xx.reshape(-1), yy.reshape(-1)]).T
    return centers

def get_cell_boundary(center, topology='rectangular'):
    if topology == 'rectangular':
        points = [np.array([[center[0]-0.5, center[1]-0.5], [center[0]-0.5, center[1]+0.5]]),
                 np.array([[center[0]-0.5, center[1]+0.5], [center[0]+0.5, center[1]+0.5]]),
                np.array([[center[0]+0.5, center[1]-0.5], [center[0]+0.5, center[1]+0.5]]),
                 np.array([[center[0]-0.5, center[1]-0.5], [center[0]+0.5, center[1]-0.5]])]
        return points
    elif topology == 'hexagonal':
        dx = 0.5
        dy = np.sqrt(3)/6
        points = [np.array([[center[0]-dx, center[1]+dy], [center[0], center[1]+2*dy]]),
                  np.array([[center[0], center[1]+2*dy], [center[0]+dx, center[1]+dy]]),
                 np.array([[center[0]+dx, center[1]-dy], [center[0]+dx, center[1]+dy]]),
                 np.array([[center[0], center[1]-2*dy], [center[0]+dx, center[1]-dy]]),
                 np.array([[center[0]-dx, center[1]-dy], [center[0], center[1]-2*dy]]),
                 np.array([[center[0]-dx, center[1]-dy], [center[0]-dx, center[1]+dy]])]
    return points

def get_manycells_boundary(centers, topology='rectangular'):
    points=[]
    for center in centers:
        points+=get_cell_boundary(center, topology)
    points_unique, counts = np.unique(np.round(np.array(points),4), axis=0, return_counts=True)
    return points_unique[counts==1]


# Now let's do the quality control.

Quality control means selecting a subset of the clusters/SOM cells where the target galaxies are well represented by the reference sample.
We evaluate the "goodness-of-reference" by the two criteria given by [arXiv:2007.15635](https://arxiv.org/pdf/2007.15635):

Quality cut 1 (QC1):

$\frac{\left|\left\langle z_{\text {spec }}\right\rangle-\left\langle Z_{\mathrm{B}}\right\rangle\right|}{\operatorname{nMAD}\left(\left\langle z_{\text {spec }}\right\rangle-\left\langle Z_{\mathrm{B}}\right\rangle\right)}>5$

This QC removes outliers in the distribution of photo-$z$.

Quality cut 2 (QC2):

$\left|\left\langle Z_B\right\rangle_{\text {spec }}-\left\langle Z_B\right\rangle_{\text {phot }}\right|>0.02$

This QC removes clusters in which the target and reference sample have very different photo-$z$, meaning the reference galaxies are not representative.

We also try to combine these two QCs (QC1+QC2).

**KiDS-Legacy modification:** the traditional approach evaluates QC1/QC2 once, on a single SOM/clustering, and hard-cuts the clusters that fail. Here we evaluate the *identical* QC1/QC2/QC1+QC2 criteria independently for each of our `N_REALIZATIONS` SOM realizations (each with its own hierarchical clustering into `n_clusters` SOM clusters), and record for every target galaxy whether its cluster passed each criterion in that realization. The **gold weight** for a given QC scheme is then the fraction of realizations in which a galaxy's cluster passed that criterion, replacing the hard `useful_clusters` cut with a continuous per-galaxy weight.

In [ ]:
def evaluate_realization_qc(som, ref_colors, target_colors, ref_true_z, ref_bpz, target_true_z, target_bpz,
                            n_clusters, n_rows, n_columns, som_split_size=1000):
    """Evaluate the QC1/QC2 criteria (arXiv:2007.15635) for one trained SOM
    realization.

    Clusters the SOM into `n_clusters` hierarchical clusters, assigns the
    reference and target samples to clusters, and computes the per-cluster
    QC1/QC2 statistics. Returns, for every TARGET (photometric) galaxy:
      baseline_flag   : 1 if the galaxy's cluster contains any reference-sample
                         (spectroscopic) members, else 0 -- this is the nominal
                         KiDS-1000 gold-flag definition (Wright et al. 2020)
      qc1_flag        : 1 if the galaxy's cluster passes QC1
      qc2_flag        : 1 if the galaxy's cluster passes QC2
      qccombined_flag : 1 if the galaxy's cluster passes QC1 AND QC2
    plus a dict of per-cluster diagnostics used for the illustrative plots.
    """
    algorithm = sc.AgglomerativeClustering(n_clusters=n_clusters, linkage='complete')
    som.cluster(algorithm)
    som_cluster_inds = som.clusters.reshape(-1)

    ref_bmu = get_bmus(som, ref_colors, som_split_size).T
    target_bmu = get_bmus(som, target_colors, som_split_size).T

    ref_pixel_coords = np.ravel_multi_index(ref_bmu, (n_columns, n_rows))
    target_pixel_coords = np.ravel_multi_index(target_bmu, (n_columns, n_rows))
    ref_clusterind = som_cluster_inds[ref_pixel_coords]
    target_clusterind = som_cluster_inds[target_pixel_coords]

    # "specZ" = true redshift, "photZ" = BPZ point estimate (Z_B)
    mean_specZ_ref = np.full(n_clusters, np.nan)
    mean_photZ_ref = np.full(n_clusters, np.nan)
    mean_photZ_target = np.full(n_clusters, np.nan)
    mean_specZ_target = np.full(n_clusters, np.nan)

    for i in range(n_clusters):
        rmask = ref_clusterind == i
        tmask = target_clusterind == i
        if np.any(rmask):
            mean_specZ_ref[i] = np.median(ref_true_z[rmask])
            mean_photZ_ref[i] = np.median(ref_bpz[rmask])
        if np.any(tmask):
            mean_photZ_target[i] = np.median(target_bpz[tmask])
            mean_specZ_target[i] = np.median(target_true_z[tmask])

    zmean_diff_cluster_qc1 = np.fabs(mean_specZ_ref - mean_photZ_target)
    zmean_diff_cluster_qc2 = np.fabs(mean_photZ_target - mean_photZ_ref)

    valid_ref = ~np.isnan(mean_specZ_ref - mean_photZ_ref)
    qc1_threshold = max(median_abs_deviation((mean_specZ_ref - mean_photZ_ref)[valid_ref]) * 5, 0)

    # baseline gold flag (nominal KiDS-1000 definition, Wright et al. 2020):
    # the cluster contains any reference-sample (spectroscopic) members --
    # this does NOT require the cluster to also contain target-sample members.
    cluster_has_ref = ~np.isnan(mean_specZ_ref)

    # QC1/QC2 additionally need a target-sample median in the cluster, since
    # the diff formulas themselves are undefined without one.
    cluster_qc_valid = cluster_has_ref & ~np.isnan(mean_photZ_target)
    cluster_good_qc1 = cluster_qc_valid & (zmean_diff_cluster_qc1 < qc1_threshold)
    cluster_good_qc2 = cluster_qc_valid & (zmean_diff_cluster_qc2 < 0.02)
    cluster_good_qccombined = cluster_good_qc1 & cluster_good_qc2

    baseline_flag = cluster_has_ref[target_clusterind].astype(float)
    qc1_flag = cluster_good_qc1[target_clusterind].astype(float)
    qc2_flag = cluster_good_qc2[target_clusterind].astype(float)
    qccombined_flag = cluster_good_qccombined[target_clusterind].astype(float)

    diagnostics = dict(
        som_cluster_inds=som_cluster_inds,
        mean_specZ_ref=mean_specZ_ref, mean_photZ_ref=mean_photZ_ref,
        mean_photZ_target=mean_photZ_target, mean_specZ_target=mean_specZ_target,
        zmean_diff_cluster_qc1=zmean_diff_cluster_qc1, zmean_diff_cluster_qc2=zmean_diff_cluster_qc2,
        qc1_threshold=qc1_threshold,
        cluster_good_qc1=cluster_good_qc1, cluster_good_qc2=cluster_good_qc2,
        cluster_good_qccombined=cluster_good_qccombined, cluster_has_ref=cluster_has_ref,
        target_clusterind=target_clusterind, ref_clusterind=ref_clusterind,
        target_pixel_coords=target_pixel_coords,
    )
    return baseline_flag, qc1_flag, qc2_flag, qccombined_flag, diagnostics

In [ ]:
%%time
n_clusters = dim ** 2 - 1

ref_true_z = ref_data.data['photometry']['redshift']
target_true_z = target_data.data['photometry']['redshift']

gold_class_baseline = np.zeros((N_REALIZATIONS, ngal_target))
gold_class_qc1 = np.zeros((N_REALIZATIONS, ngal_target))
gold_class_qc2 = np.zeros((N_REALIZATIONS, ngal_target))
gold_class_qccombined = np.zeros((N_REALIZATIONS, ngal_target))

fiducial_diagnostics = None

for j, som_model_j in enumerate(som_models):
    som_j = som_model_j['som']
    (gold_class_baseline[j], gold_class_qc1[j], gold_class_qc2[j],
     gold_class_qccombined[j], diagnostics_j) = evaluate_realization_qc(
        som_j, ref_colors, target_colors, ref_true_z, bpz_spec, target_true_z, bpz_phot,
        n_clusters, n_rows, n_columns)
    if j == 0:
        fiducial_diagnostics = diagnostics_j
    print(f"Realization {j + 1}/{N_REALIZATIONS}: "
          f"{int(diagnostics_j['cluster_good_qc1'].sum())} QC1-good, "
          f"{int(diagnostics_j['cluster_good_qc2'].sum())} QC2-good, "
          f"{int(diagnostics_j['cluster_good_qccombined'].sum())} QC1+QC2-good "
          f"out of {n_clusters} clusters")

In [ ]:
gold_weight_baseline = gold_class_baseline.mean(axis=0)
gold_weight_qc1 = gold_class_qc1.mean(axis=0)
gold_weight_qc2 = gold_class_qc2.mean(axis=0)
gold_weight_qccombined = gold_class_qccombined.mean(axis=0)

# add the gold weights to the target (photometric) data as new per-galaxy entries
target_data.data['photometry']['gold_weight'] = gold_weight_baseline
target_data.data['photometry']['gold_weight_qc1'] = gold_weight_qc1
target_data.data['photometry']['gold_weight_qc2'] = gold_weight_qc2
target_data.data['photometry']['gold_weight_qccombined'] = gold_weight_qccombined

print(f"baseline (has spectroscopic reference): mean gold weight = {gold_weight_baseline.mean():.3f}")
print(f"QC1: mean gold weight = {gold_weight_qc1.mean():.3f}, "
      f"fraction with weight==0 = {(gold_weight_qc1 == 0).mean():.3f}")
print(f"QC2: mean gold weight = {gold_weight_qc2.mean():.3f}, "
      f"fraction with weight==0 = {(gold_weight_qc2 == 0).mean():.3f}")
print(f"QC1+QC2: mean gold weight = {gold_weight_qccombined.mean():.3f}, "
      f"fraction with weight==0 = {(gold_weight_qccombined == 0).mean():.3f}")

Now we plot the SOM grid (fiducial realization) color-coded by the average true redshift of the target sample (left panel) and the reference sample (right panel), to show the reference sample is indeed representative of the target galaxies' redshifts.

In [ ]:
# SOM.clusters already reflects the n_clusters=1000 clustering computed for
# realization 0 inside the evaluate_realization_qc loop above -- no need to re-cluster.
mean_photZ_ref = fiducial_diagnostics['mean_photZ_ref'][fiducial_diagnostics['som_cluster_inds']]
mean_specZ_ref = fiducial_diagnostics['mean_specZ_ref'][fiducial_diagnostics['som_cluster_inds']]
mean_photZ_target = fiducial_diagnostics['mean_photZ_target'][fiducial_diagnostics['som_cluster_inds']]
mean_specZ_target = fiducial_diagnostics['mean_specZ_target'][fiducial_diagnostics['som_cluster_inds']]

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12,5))
plot_som(ax[0], mean_specZ_target.reshape(dim, dim), grid_type=grid_type, colormap=cm.coolwarm, cbar_name='mean true redshift of the target sample', vmin=bin_low, vmax=bin_high)
ax[0].set_title('Target sample (fiducial realization)')

plot_som(ax[1], mean_specZ_ref.reshape(dim, dim), grid_type=grid_type, colormap=cm.coolwarm, cbar_name='mean true redshift of the reference sample', vmin=bin_low, vmax=bin_high)

ax[1].set_title('Reference sample (fiducial realization)')
plot_cluster_boundaries(ax[0], SOM, n_clusters)

Here is the QC1/QC2 diagnostic map for the **fiducial (single) realization**, with the good/bad cluster boundaries overlaid:

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12,5))

qc1_map = (fiducial_diagnostics['zmean_diff_cluster_qc1'] / max(fiducial_diagnostics['qc1_threshold'], 1e-12))[fiducial_diagnostics['som_cluster_inds']]
plot_som(ax[0], qc1_map.reshape(dim, dim), grid_type=grid_type, colormap=cm.coolwarm,
         cbar_name=r'$\frac{\left|\left\langle z_{\text {spec }}\right\rangle-\left\langle Z_{\mathrm{B}}\right\rangle\right|}{\operatorname{nMAD}\left(\left\langle z_{\text {spec }}\right\rangle-\left\langle Z_{\mathrm{B}}\right\rangle\right)\times 5}$', vmin=0, vmax=1)
plot_cluster_boundaries(ax[0], SOM, n_clusters, cluster_inds=np.where(fiducial_diagnostics['cluster_good_qc1'])[0])
ax[0].set_title('QC1 (fiducial realization)')

qc2_map = (fiducial_diagnostics['zmean_diff_cluster_qc2'] / 0.02)[fiducial_diagnostics['som_cluster_inds']]
plot_som(ax[1], qc2_map.reshape(dim, dim), grid_type=grid_type, colormap=cm.coolwarm,
         cbar_name=r'$\left|\left\langle Z_B\right\rangle_{\text {spec }}-\left\langle Z_B\right\rangle_{\text {phot }}\right|/0.02$', vmin=0, vmax=1)
plot_cluster_boundaries(ax[1], SOM, n_clusters, cluster_inds=np.where(fiducial_diagnostics['cluster_good_qc2'])[0])
ax[1].set_title('QC2 (fiducial realization)')

## Diagnostic SOM map of the gold weight for QC1 and QC2

Instead of a single realization's hard pass/fail, let's map the **continuous QC1/QC2 gold weight** (the fraction of the `N_REALIZATIONS` realizations in which each target galaxy's cluster passed QC1/QC2) onto the SOM grid, in the same style as the single-realization diagnostic maps above: we average the per-galaxy gold weight within each of the fiducial realization's own SOM clusters, then broadcast that per-cluster average back onto every cell belonging to the cluster. The black outlines again show the fiducial (single-realization) good-cluster boundaries for comparison, so it's easy to see clusters just outside those hard boundaries that nonetheless have substantial QC gold weight, and vice versa.

In [ ]:
target_clusterind_fid = fiducial_diagnostics['target_clusterind']
som_cluster_inds_fid = fiducial_diagnostics['som_cluster_inds']

qc1_weight_per_cluster = np.full(n_clusters, np.nan)
qc2_weight_per_cluster = np.full(n_clusters, np.nan)
for i in range(n_clusters):
    tmask = target_clusterind_fid == i
    if np.any(tmask):
        qc1_weight_per_cluster[i] = np.mean(gold_weight_qc1[tmask])
        qc2_weight_per_cluster[i] = np.mean(gold_weight_qc2[tmask])

qc1_weight_map = qc1_weight_per_cluster[som_cluster_inds_fid].reshape(dim, dim)
qc2_weight_map = qc2_weight_per_cluster[som_cluster_inds_fid].reshape(dim, dim)

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12,5))

plot_som(ax[0], qc1_weight_map, grid_type=grid_type, colormap=cm.coolwarm, cbar_name='QC1 gold weight', vmin=0, vmax=1)
plot_cluster_boundaries(ax[0], SOM, n_clusters, cluster_inds=np.where(fiducial_diagnostics['cluster_good_qc1'])[0])
ax[0].set_title(f'QC1 gold weight (after {N_REALIZATIONS} realizations)')

plot_som(ax[1], qc2_weight_map, grid_type=grid_type, colormap=cm.coolwarm, cbar_name='QC2 gold weight', vmin=0, vmax=1)
plot_cluster_boundaries(ax[1], SOM, n_clusters, cluster_inds=np.where(fiducial_diagnostics['cluster_good_qc2'])[0])
ax[1].set_title(f'QC2 gold weight (after {N_REALIZATIONS} realizations)')

Now that we have illustrated what exactly we have constructed, let's use the SOM to predict the redshift distribution for the target sample.
We first summarize the gold-weight baseline (has spectroscopic reference) case, then the three gold-weighted QC cases (QC1, QC2, QC1+QC2).

Note that we have removed the 'photometry' group, we will specify the `phot_groupname` as "" in the parameters below.<br>
As before, let us specify our initialization params for the `SOMocluSummarizer` stage, including:<br>
`model`: the fiducial trained SOM model (realization 0)<br>
`hdf5_groupname` (str): hdf5 group for our photometric data<br>
`objid_name` (str): string specifying the name of the ID column<br>
`spec_groupname` (str): hdf5 group for the spectroscopic data<br>
`nzbins` (int): number of bins to use in our histogram ensemble<br>
`n_clusters` (int): number of hierarchical clusters (1000)<br>
`n_samples` (int): number of bootstrap samples to generate<br>
`output` / `single_NZ` / `uncovered_cluster_file` / `cellid_output`: output file names<br>
**`phot_weightcol` (str): the KiDS-Legacy gold-weight column for this case (`gold_weight`, `gold_weight_qc1`, `gold_weight_qc2`, or `gold_weight_qccombined`) -- this replaces a hard `useful_clusters` cut.** Every target galaxy is still included in the summarization, but galaxies whose cluster only sometimes passes a QC criterion across the 10 realizations are down-weighted rather than being fully kept or fully discarded.

In [ ]:
def get_cont_hist(data, bins, weights=None):
    hist, bin_edge = np.histogram(data, bins=bins, density=True, weights=weights)
    return hist, (bin_edge[1:]+bin_edge[:-1])/2

def weighted_mean_std(z, w):
    w = np.asarray(w)
    mean = np.average(z, weights=w)
    std = np.sqrt(np.average((z - mean) ** 2, weights=w))
    return mean, std

zbin_edges = np.linspace(0, 3, 101)

In [ ]:
summ_dict = dict(model=model, hdf5_groupname='photometry',
                 spec_groupname='photometry', nzbins=101, n_samples=25,
                 output='KL_SOM_ensemble.hdf5', single_NZ='KL_fiducial_SOMoclu_NZ.hdf5',
                 n_clusters=n_clusters,
                 uncovered_cluster_file='KL_all_uncovered_cells.hdf5',
                 objid_name='id',
                 cellid_output='KL_output_cellIDs.hdf5',
                 phot_weightcol='gold_weight')

som_summarizer = SOMocluSummarizer.make_stage(name='SOMoclu_summarizer_kl', **summ_dict)
som_summarizer.summarize(target_data, ref_data)

fid_ens = qp.read('KL_fiducial_SOMoclu_NZ.hdf5')
boot_ens = qp.read('KL_SOM_ensemble.hdf5')

target_nz_hist, zbin = get_cont_hist(target_data.data['photometry']['redshift'], zbin_edges, weights=gold_weight_baseline)
som_nz_hist = np.squeeze(fid_ens.pdf(zbin))

full_ens = qp.read("KL_SOM_ensemble.hdf5")
full_means = full_ens.mean().flatten()
full_stds = full_ens.std().flatten()
true_full_mean, true_full_std = weighted_mean_std(target_data.data['photometry']['redshift'], gold_weight_baseline)

print('===========This is the gold-weight baseline case==================')
print("The mean redshift of the SOM ensemble is: "+str(round(np.mean(full_means),4)) + '+-' + str(round(np.std(full_means),4)))
print("The mean redshift of the real data is: "+str(round(true_full_mean,4)))
print("The bias of mean redshift is:"+str(round(np.mean(full_means)-true_full_mean,4)) + '+-' + str(round(np.std(full_means),4)))

In [ ]:
summ_dict_qc1 = dict(model=model, hdf5_groupname='photometry',
                 spec_groupname='photometry', nzbins=101, n_samples=25,
                 output='KL_SOM_ensemble_qc1.hdf5', single_NZ='KL_fiducial_SOMoclu_NZ_qc1.hdf5',
                 n_clusters=n_clusters,
                 uncovered_cluster_file='KL_all_uncovered_cells_qc1.hdf5',
                 objid_name='id',
                 cellid_output='KL_output_cellIDs_qc1.hdf5',
                 phot_weightcol='gold_weight_qc1')

som_summarizer_qc1 = SOMocluSummarizer.make_stage(name='SOMoclu_summarizer_qc1_kl', **summ_dict_qc1)
som_summarizer_qc1.summarize(target_data, ref_data)

fid_ens_qc1 = qp.read("KL_fiducial_SOMoclu_NZ_qc1.hdf5")
boot_ens_qc1 = qp.read('KL_SOM_ensemble_qc1.hdf5')
som_nz_hist_qc1 = np.squeeze(fid_ens_qc1.pdf(zbin))
target_nz_hist_qc1_true, zbin = get_cont_hist(target_data.data['photometry']['redshift'], zbin_edges, weights=gold_weight_qc1)

full_ens_qc1 = qp.read("KL_SOM_ensemble_qc1.hdf5")
full_means_qc1 = full_ens_qc1.mean().flatten()
full_stds_qc1 = full_ens_qc1.std().flatten()
true_full_mean_qc1, true_full_std_qc1 = weighted_mean_std(target_data.data['photometry']['redshift'], gold_weight_qc1)

print('\n\n===========This is the QC1 gold-weight case==================')
print("The mean redshift of the SOM ensemble is: "+str(round(np.mean(full_means_qc1),4)) + '+-' + str(round(np.std(full_means_qc1),4)))
print("The mean redshift of the real data is: "+str(round(true_full_mean_qc1,4)))
print("The bias of mean redshift is:"+str(round(np.mean(full_means_qc1)-true_full_mean_qc1,4)) + '+-' + str(round(np.std(full_means_qc1),4)))

In [ ]:
summ_dict_qc2 = dict(model=model, hdf5_groupname='photometry',
                 spec_groupname='photometry', nzbins=101, n_samples=25,
                 output='KL_SOM_ensemble_qc2.hdf5', single_NZ='KL_fiducial_SOMoclu_NZ_qc2.hdf5',
                 n_clusters=n_clusters,
                 uncovered_cluster_file='KL_all_uncovered_cells_qc2.hdf5',
                 objid_name='id',
                 cellid_output='KL_output_cellIDs_qc2.hdf5',
                 phot_weightcol='gold_weight_qc2')

som_summarizer_qc2 = SOMocluSummarizer.make_stage(name='SOMoclu_summarizer_qc2_kl', **summ_dict_qc2)
som_summarizer_qc2.summarize(target_data, ref_data)

fid_ens_qc2 = qp.read("KL_fiducial_SOMoclu_NZ_qc2.hdf5")
boot_ens_qc2 = qp.read('KL_SOM_ensemble_qc2.hdf5')
som_nz_hist_qc2 = np.squeeze(fid_ens_qc2.pdf(zbin))
target_nz_hist_qc2_true, zbin = get_cont_hist(target_data.data['photometry']['redshift'], zbin_edges, weights=gold_weight_qc2)

full_ens_qc2 = qp.read("KL_SOM_ensemble_qc2.hdf5")
full_means_qc2 = full_ens_qc2.mean().flatten()
full_stds_qc2 = full_ens_qc2.std().flatten()
true_full_mean_qc2, true_full_std_qc2 = weighted_mean_std(target_data.data['photometry']['redshift'], gold_weight_qc2)

print("\n\n===========This is the QC2 gold-weight case==================")
print("The mean redshift of the SOM ensemble is: "+str(round(np.mean(full_means_qc2),4)) + '+-' + str(round(np.std(full_means_qc2),4)))
print("The mean redshift of the real data is: "+str(round(true_full_mean_qc2,4)))
print("The bias of mean redshift is:"+str(round(np.mean(full_means_qc2)-true_full_mean_qc2,4)) + '+-' + str(round(np.std(full_means_qc2),4)))

In [ ]:
summ_dict_qccombined = dict(model=model, hdf5_groupname='photometry',
                 spec_groupname='photometry', nzbins=101, n_samples=25,
                 output='KL_SOM_ensemble_qccombined.hdf5', single_NZ='KL_fiducial_SOMoclu_NZ_qccombined.hdf5',
                 n_clusters=n_clusters,
                 uncovered_cluster_file='KL_all_uncovered_cells_qccombined.hdf5',
                 objid_name='id',
                 cellid_output='KL_output_cellIDs_qccombined.hdf5',
                 phot_weightcol='gold_weight_qccombined')

som_summarizer_qccombined = SOMocluSummarizer.make_stage(name='SOMoclu_summarizer_qccombined_kl', **summ_dict_qccombined)
som_summarizer_qccombined.summarize(target_data, ref_data)

fid_ens_qccombined = qp.read("KL_fiducial_SOMoclu_NZ_qccombined.hdf5")
boot_ens_qccombined = qp.read('KL_SOM_ensemble_qccombined.hdf5')
som_nz_hist_qccombined = np.squeeze(fid_ens_qccombined.pdf(zbin))
target_nz_hist_qccombined_true, zbin = get_cont_hist(target_data.data['photometry']['redshift'], zbin_edges, weights=gold_weight_qccombined)

full_ens_qccombined = qp.read("KL_SOM_ensemble_qccombined.hdf5")
full_means_qccombined = full_ens_qccombined.mean().flatten()
full_stds_qccombined = full_ens_qccombined.std().flatten()
true_full_mean_qccombined, true_full_std_qccombined = weighted_mean_std(target_data.data['photometry']['redshift'], gold_weight_qccombined)

print("\n\n===========This is the QC1+QC2 gold-weight case==================")
print("The mean redshift of the SOM ensemble is: "+str(round(np.mean(full_means_qccombined),4)) + '+-' + str(round(np.std(full_means_qccombined),4)))
print("The mean redshift of the real data is: "+str(round(true_full_mean_qccombined,4)))
print("The bias of mean redshift is:"+str(round(np.mean(full_means_qccombined)-true_full_mean_qccombined,4)) + '+-' + str(round(np.std(full_means_qccombined),4)))

## Gold flag (KiDS-1000): baseline, QC1, QC2, and QC1+QC2

For comparison, let's also build the N(z) using the traditional **gold flag** (KiDS-1000-style, Wright et al. 2020) definition of the baseline, QC1, QC2, and QC1+QC2 cases: a hard cut on the clusters that pass each criterion under the single fiducial (realization 0) SOM, using RAIL's `useful_clusters` mechanism rather than a continuous `phot_weightcol` -- i.e. a galaxy's cluster either fully passes or fully fails, based on that one SOM training. These gold-flag cases are added to the box plot below alongside the gold-weight cases, to directly compare the traditional hard cut against the KiDS-Legacy continuous weight.

In [ ]:
summ_dict_baseline_flag = dict(model=model, hdf5_groupname='photometry',
                 spec_groupname='photometry', nzbins=101, n_samples=25,
                 output='KL_SOM_ensemble_baseline_flag.hdf5', single_NZ='KL_fiducial_SOMoclu_NZ_baseline_flag.hdf5',
                 n_clusters=n_clusters,
                 uncovered_cluster_file='KL_all_uncovered_cells_baseline_flag.hdf5',
                 objid_name='id',
                 cellid_output='KL_output_cellIDs_baseline_flag.hdf5',
                 useful_clusters=np.where(fiducial_diagnostics['cluster_has_ref'])[0])

som_summarizer_baseline_flag = SOMocluSummarizer.make_stage(name='SOMoclu_summarizer_baseline_flag_kl', **summ_dict_baseline_flag)
som_summarizer_baseline_flag.summarize(target_data, ref_data)

fid_ens_baseline_flag = qp.read("KL_fiducial_SOMoclu_NZ_baseline_flag.hdf5")
boot_ens_baseline_flag = qp.read('KL_SOM_ensemble_baseline_flag.hdf5')
som_nz_hist_baseline_flag = np.squeeze(fid_ens_baseline_flag.pdf(zbin))

# gold_class_baseline[0] is exactly the fiducial realization's own binary
# 'has spectroscopic reference' flag per target galaxy (row 0 of the
# N_REALIZATIONS ensemble computed above)
baseline_flag = gold_class_baseline[0]
target_nz_hist_baseline_flag_true, _ = get_cont_hist(target_data.data['photometry']['redshift'], zbin_edges, weights=baseline_flag)

full_ens_baseline_flag = qp.read("KL_SOM_ensemble_baseline_flag.hdf5")
full_means_baseline_flag = full_ens_baseline_flag.mean().flatten()
full_stds_baseline_flag = full_ens_baseline_flag.std().flatten()
true_full_mean_baseline_flag, true_full_std_baseline_flag = weighted_mean_std(target_data.data['photometry']['redshift'], baseline_flag)

print('===========This is the baseline gold-flag (KiDS-1000) case==================')
print("The mean redshift of the SOM ensemble is: "+str(round(np.mean(full_means_baseline_flag),4)) + '+-' + str(round(np.std(full_means_baseline_flag),4)))
print("The mean redshift of the real data is: "+str(round(true_full_mean_baseline_flag,4)))
print("The bias of mean redshift is:"+str(round(np.mean(full_means_baseline_flag)-true_full_mean_baseline_flag,4)) + '+-' + str(round(np.std(full_means_baseline_flag),4)))

In [ ]:
summ_dict_qc1_flag = dict(model=model, hdf5_groupname='photometry',
                 spec_groupname='photometry', nzbins=101, n_samples=25,
                 output='KL_SOM_ensemble_qc1_flag.hdf5', single_NZ='KL_fiducial_SOMoclu_NZ_qc1_flag.hdf5',
                 n_clusters=n_clusters,
                 uncovered_cluster_file='KL_all_uncovered_cells_qc1_flag.hdf5',
                 objid_name='id',
                 cellid_output='KL_output_cellIDs_qc1_flag.hdf5',
                 useful_clusters=np.where(fiducial_diagnostics['cluster_good_qc1'])[0])

som_summarizer_qc1_flag = SOMocluSummarizer.make_stage(name='SOMoclu_summarizer_qc1_flag_kl', **summ_dict_qc1_flag)
som_summarizer_qc1_flag.summarize(target_data, ref_data)

fid_ens_qc1_flag = qp.read("KL_fiducial_SOMoclu_NZ_qc1_flag.hdf5")
boot_ens_qc1_flag = qp.read('KL_SOM_ensemble_qc1_flag.hdf5')
som_nz_hist_qc1_flag = np.squeeze(fid_ens_qc1_flag.pdf(zbin))

# gold_class_qc1[0] is exactly the fiducial realization's own binary QC1 flag
# per target galaxy (row 0 of the N_REALIZATIONS ensemble computed above)
qc1_flag = gold_class_qc1[0]
target_nz_hist_qc1_flag_true, _ = get_cont_hist(target_data.data['photometry']['redshift'], zbin_edges, weights=qc1_flag)

full_ens_qc1_flag = qp.read("KL_SOM_ensemble_qc1_flag.hdf5")
full_means_qc1_flag = full_ens_qc1_flag.mean().flatten()
full_stds_qc1_flag = full_ens_qc1_flag.std().flatten()
true_full_mean_qc1_flag, true_full_std_qc1_flag = weighted_mean_std(target_data.data['photometry']['redshift'], qc1_flag)

print('===========This is the QC1 gold-flag (KiDS-1000) case==================')
print("The mean redshift of the SOM ensemble is: "+str(round(np.mean(full_means_qc1_flag),4)) + '+-' + str(round(np.std(full_means_qc1_flag),4)))
print("The mean redshift of the real data is: "+str(round(true_full_mean_qc1_flag,4)))
print("The bias of mean redshift is:"+str(round(np.mean(full_means_qc1_flag)-true_full_mean_qc1_flag,4)) + '+-' + str(round(np.std(full_means_qc1_flag),4)))

In [ ]:
summ_dict_qc2_flag = dict(model=model, hdf5_groupname='photometry',
                 spec_groupname='photometry', nzbins=101, n_samples=25,
                 output='KL_SOM_ensemble_qc2_flag.hdf5', single_NZ='KL_fiducial_SOMoclu_NZ_qc2_flag.hdf5',
                 n_clusters=n_clusters,
                 uncovered_cluster_file='KL_all_uncovered_cells_qc2_flag.hdf5',
                 objid_name='id',
                 cellid_output='KL_output_cellIDs_qc2_flag.hdf5',
                 useful_clusters=np.where(fiducial_diagnostics['cluster_good_qc2'])[0])

som_summarizer_qc2_flag = SOMocluSummarizer.make_stage(name='SOMoclu_summarizer_qc2_flag_kl', **summ_dict_qc2_flag)
som_summarizer_qc2_flag.summarize(target_data, ref_data)

fid_ens_qc2_flag = qp.read("KL_fiducial_SOMoclu_NZ_qc2_flag.hdf5")
boot_ens_qc2_flag = qp.read('KL_SOM_ensemble_qc2_flag.hdf5')
som_nz_hist_qc2_flag = np.squeeze(fid_ens_qc2_flag.pdf(zbin))

qc2_flag = gold_class_qc2[0]
target_nz_hist_qc2_flag_true, _ = get_cont_hist(target_data.data['photometry']['redshift'], zbin_edges, weights=qc2_flag)

full_ens_qc2_flag = qp.read("KL_SOM_ensemble_qc2_flag.hdf5")
full_means_qc2_flag = full_ens_qc2_flag.mean().flatten()
full_stds_qc2_flag = full_ens_qc2_flag.std().flatten()
true_full_mean_qc2_flag, true_full_std_qc2_flag = weighted_mean_std(target_data.data['photometry']['redshift'], qc2_flag)

print('===========This is the QC2 gold-flag (KiDS-1000) case==================')
print("The mean redshift of the SOM ensemble is: "+str(round(np.mean(full_means_qc2_flag),4)) + '+-' + str(round(np.std(full_means_qc2_flag),4)))
print("The mean redshift of the real data is: "+str(round(true_full_mean_qc2_flag,4)))
print("The bias of mean redshift is:"+str(round(np.mean(full_means_qc2_flag)-true_full_mean_qc2_flag,4)) + '+-' + str(round(np.std(full_means_qc2_flag),4)))

In [ ]:
summ_dict_qccombined_flag = dict(model=model, hdf5_groupname='photometry',
                 spec_groupname='photometry', nzbins=101, n_samples=25,
                 output='KL_SOM_ensemble_qccombined_flag.hdf5', single_NZ='KL_fiducial_SOMoclu_NZ_qccombined_flag.hdf5',
                 n_clusters=n_clusters,
                 uncovered_cluster_file='KL_all_uncovered_cells_qccombined_flag.hdf5',
                 objid_name='id',
                 cellid_output='KL_output_cellIDs_qccombined_flag.hdf5',
                 useful_clusters=np.where(fiducial_diagnostics['cluster_good_qccombined'])[0])

som_summarizer_qccombined_flag = SOMocluSummarizer.make_stage(name='SOMoclu_summarizer_qccombined_flag_kl', **summ_dict_qccombined_flag)
som_summarizer_qccombined_flag.summarize(target_data, ref_data)

fid_ens_qccombined_flag = qp.read("KL_fiducial_SOMoclu_NZ_qccombined_flag.hdf5")
boot_ens_qccombined_flag = qp.read('KL_SOM_ensemble_qccombined_flag.hdf5')
som_nz_hist_qccombined_flag = np.squeeze(fid_ens_qccombined_flag.pdf(zbin))

qccombined_flag = gold_class_qccombined[0]
target_nz_hist_qccombined_flag_true, _ = get_cont_hist(target_data.data['photometry']['redshift'], zbin_edges, weights=qccombined_flag)

full_ens_qccombined_flag = qp.read("KL_SOM_ensemble_qccombined_flag.hdf5")
full_means_qccombined_flag = full_ens_qccombined_flag.mean().flatten()
full_stds_qccombined_flag = full_ens_qccombined_flag.std().flatten()
true_full_mean_qccombined_flag, true_full_std_qccombined_flag = weighted_mean_std(target_data.data['photometry']['redshift'], qccombined_flag)

print('===========This is the QC1+QC2 gold-flag (KiDS-1000) case==================')
print("The mean redshift of the SOM ensemble is: "+str(round(np.mean(full_means_qccombined_flag),4)) + '+-' + str(round(np.std(full_means_qccombined_flag),4)))
print("The mean redshift of the real data is: "+str(round(true_full_mean_qccombined_flag,4)))
print("The bias of mean redshift is:"+str(round(np.mean(full_means_qccombined_flag)-true_full_mean_qccombined_flag,4)) + '+-' + str(round(np.std(full_means_qccombined_flag),4)))

Now we plot the true/calibrated redshift distributions of the four gold-weighted cases. Each panel's title shows the effective fraction of the photometric weight retained relative to the baseline case, via `neff_p_to_neff`.

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(24,12))
ax = ax.flatten()
ax[0].set_xlabel("redshift", fontsize=15)
ax[0].set_ylabel("N(z)", fontsize=15)
ax[0].set_title('gold-weight baseline (has spectroscopic reference)')
ax[0].plot(zbin, target_nz_hist, label='True N(z)')
ax[0].plot(zbin, som_nz_hist,  color='C1',label='SOM N(z)')

for i in range(boot_ens.npdf):
    pdf = np.squeeze(boot_ens[i].pdf(zbin))
    ax[0].plot(zbin, pdf, color='C1',zorder=0, alpha=0.2)

ax[1].set_title(f'QC1 gold-weight: {round(som_summarizer_qc1.neff_p_to_neff / som_summarizer.neff_p_to_neff*100, 2)}% effective weight retained')
ax[1].plot(zbin, target_nz_hist_qc1_true, color='C0',label='True N(z), QC1 gold-weighted')
ax[1].plot(zbin, som_nz_hist_qc1, color='C1',label='SOM N(z), QC1 gold-weighted')
ax[1].set_xlabel("redshift", fontsize=15)
ax[1].set_ylabel("N(z)", fontsize=15)
for i in range(boot_ens_qc1.npdf):
    pdf = np.squeeze(boot_ens_qc1[i].pdf(zbin))
    ax[1].plot(zbin, pdf, color='C1',zorder=0, alpha=0.2)

ax[2].set_title(f'QC2 gold-weight: {round(som_summarizer_qc2.neff_p_to_neff / som_summarizer.neff_p_to_neff*100, 2)}% effective weight retained')
ax[2].plot(zbin, target_nz_hist_qc2_true, color='C0',label='True N(z), QC2 gold-weighted')
ax[2].plot(zbin, som_nz_hist_qc2, color='C1',label='SOM N(z), QC2 gold-weighted')
ax[2].set_xlabel("redshift", fontsize=15)
ax[2].set_ylabel("N(z)", fontsize=15)
for i in range(boot_ens_qc2.npdf):
    pdf = np.squeeze(boot_ens_qc2[i].pdf(zbin))
    ax[2].plot(zbin, pdf, color='C1',zorder=0, alpha=0.2)

ax[3].set_title(f'QC1+QC2 gold-weight: {round(som_summarizer_qccombined.neff_p_to_neff / som_summarizer.neff_p_to_neff*100, 2)}% effective weight retained')
ax[3].plot(zbin, target_nz_hist_qccombined_true, color='C0',label='True N(z), QC1+2 gold-weighted')
ax[3].plot(zbin, som_nz_hist_qccombined, color='C1',label='SOM N(z), QC1+2 gold-weighted')
ax[3].set_xlabel("redshift", fontsize=15)
ax[3].set_ylabel("N(z)", fontsize=15)
for i in range(boot_ens_qccombined.npdf):
    pdf = np.squeeze(boot_ens_qccombined[i].pdf(zbin))
    ax[3].plot(zbin, pdf, color='C1',zorder=0, alpha=0.2)

ax[0].legend(fontsize=15)
ax[1].legend(fontsize=15)
ax[2].legend(fontsize=15)
ax[3].legend(fontsize=15)
ax[0].set_xlim(0,1.5)
ax[1].set_xlim(0,1.5)
ax[2].set_xlim(0,1.5)
ax[3].set_xlim(0,1.5)

The following box plot summarizes the bias in the redshift estimation for the four gold-weight cases (baseline, QC1, QC2, QC1+QC2) alongside their four gold-flag (KiDS-1000, single fiducial realization) counterparts, with the two schemes shown in different box colors for easy comparison:

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

data = [full_means_qccombined_flag-true_full_mean_qccombined_flag,
        full_means_qccombined-true_full_mean_qccombined,
        full_means_qc2_flag-true_full_mean_qc2_flag,
        full_means_qc2-true_full_mean_qc2,
        full_means_qc1_flag-true_full_mean_qc1_flag,
        full_means_qc1-true_full_mean_qc1,
        full_means_baseline_flag-true_full_mean_baseline_flag,
        full_means-true_full_mean]

# if a case's mean bias is negative, plot |bias| instead and mark that box
# (and its whiskers/caps) with a dashed outline so the sign flip stays visible
is_negative = [np.mean(d) < 0 for d in data]
plot_data = [np.abs(d) if neg else d for d, neg in zip(data, is_negative)]

# data alternates gold-flag, gold-weight, gold-flag, gold-weight, ...
flag_color = 'goldenrod'
weight_color = 'steelblue'
box_colors = [flag_color, weight_color] * (len(data) // 2)

fig, ax = plt.subplots(1,1, figsize=(8,6))
bxp = ax.boxplot(plot_data, orientation='horizontal', patch_artist=True)
for i, (patch, color, neg) in enumerate(zip(bxp['boxes'], box_colors, is_negative)):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
    if neg:
        patch.set_linestyle('dashed')
        patch.set_edgecolor('k')
        patch.set_linewidth(1.5)
        for element in bxp['whiskers'][2*i:2*i+2] + bxp['caps'][2*i:2*i+2]:
            element.set_linestyle('dashed')

ax.axvline(0,1,0, color='C0')
ax.set_xlabel(r'$\Delta\left\langle z \right\rangle$ (dashed boxes: mean bias is negative, $|\Delta\left\langle z \right\rangle|$ shown)')
ax.set_yticklabels(['SOM N(z), QC1+QC2 (gold-flag)',
                     'SOM N(z), QC1+QC2 (gold-weight)',
                     'SOM N(z), QC2 (gold-flag)',
                     'SOM N(z), QC2 (gold-weight)',
                     'SOM N(z), QC1 (gold-flag)',
                     'SOM N(z), QC1 (gold-weight)',
                     'SOM N(z), baseline (gold-flag)',
                     'SOM N(z), baseline (gold-weight)'])
ax.legend(handles=[Patch(facecolor=flag_color, alpha=0.7, label='gold flag (KiDS-1000)'),
                    Patch(facecolor=weight_color, alpha=0.7, label='gold weight (KiDS-Legacy)'),
                    Line2D([0], [0], color='k', linestyle='dashed', lw=1.5, label='mean bias negative (|value| shown)')],
          loc='upper right', fontsize=9)

## How does cluster granularity affect the calibration?

This last section is unrelated to the QC1/QC2 criteria above -- it's a general exploration of how the *number* of SOM clusters affects the bias, scatter, and effective spectroscopic representation, using our fiducial (realization 0) SOM with no QC/gold-weighting applied (i.e. the plain baseline).

In [ ]:
n_clusterss = np.linspace(50, 1500, 10, dtype=int)

true_full_mean_scan = np.mean(target_data.data['photometry']['redshift'])
true_full_std_scan = np.std(target_data.data['photometry']['redshift'])
mu_diff = np.zeros(n_clusterss.size)
means_diff = np.zeros((n_clusterss.size, 50))

std_diff_mean = np.zeros(n_clusterss.size)
neff_p_to_neff_scan = np.zeros(n_clusterss.size)
std_diff = np.zeros((n_clusterss.size, 50))
for i, n_clusters_ in enumerate(n_clusterss):
    summ_dict_scan = dict(model=model, hdf5_groupname='photometry',
                 spec_groupname='photometry', nzbins=101, n_samples=50,
                 output='KL_SOM_ensemble_scan.hdf5', single_NZ='KL_fiducial_SOMoclu_NZ_scan.hdf5',
                 n_clusters=n_clusters_,
                 uncovered_cluster_file='KL_all_uncovered_cells_scan.hdf5',
                 objid_name='id',
                 cellid_output='KL_output_cellIDs_scan.hdf5')
    som_summarizer_scan = SOMocluSummarizer.make_stage(name=f'SOMoclu_summarizer_scan_kl_{i}', **summ_dict_scan)
    som_summarizer_scan.summarize(target_data, ref_data)

    full_ens_scan = qp.read("KL_SOM_ensemble_scan.hdf5")
    full_means_scan = full_ens_scan.mean().flatten()
    full_stds_scan = full_ens_scan.std().flatten()

    # mean and width of bootstraps
    mu_diff[i] = np.mean(full_means_scan) - true_full_mean_scan
    means_diff[i] = full_means_scan - true_full_mean_scan

    std_diff_mean[i] = np.mean(full_stds_scan) - true_full_std_scan
    std_diff[i] = full_stds_scan - true_full_std_scan
    neff_p_to_neff_scan[i] = som_summarizer_scan.neff_p_to_neff

In [ ]:
fig, axes = plt.subplots(ncols=3, nrows=1, figsize=(20,5))

axes[0].plot(n_clusterss, mu_diff, lw=1, color='k')
axes[0].axhline(0,1,0)
axes[0].set_xlabel('Number of clusters')
axes[0].set_ylabel(r'$\left\langle z \right\rangle - \left\langle z \right\rangle_{\mathrm{true}}$')

axes[1].plot(n_clusterss, std_diff_mean, lw=1, color='k')
axes[1].axhline(0,1,0)

axes[1].set_xlabel('Number of clusters')
axes[1].set_ylabel(r'$\mathrm{std}(z) - \mathrm{std}(z)_{\mathrm{true}}$')

axes[2].plot(n_clusterss, neff_p_to_neff_scan*100, lw=1, color='k')

axes[2].set_xlabel('Number of clusters')
axes[2].set_ylabel(r'$n_{\mathrm{eff}}\'/n_{\mathrm{eff}}$(%)')

Compared to a single-SOM, hard `useful_clusters` cut, the gold-weight scheme trades extra compute (training `N_REALIZATIONS` SOMs and re-evaluating QC1/QC2 on each) for a per-galaxy QC weight that is less sensitive to the randomness of any one SOM training, following Stölzner et al. 2025 ([arXiv:2503.19440](https://arxiv.org/abs/2503.19440)) and Wright et al. 2025 ([arXiv:2503.19441](https://arxiv.org/abs/2503.19441)). The QC1/QC2 criteria themselves -- and the gold-weight fraction they're evaluated against -- are exactly the ones from Wright et al. 2020 ([arXiv:2007.15635](https://arxiv.org/pdf/2007.15635)).